# Day 16 — Data Ingestion & I/O Optimization

This notebook covers Optimized File Ingestion (`CSV`, `Excel`, `JSON`), Schema Enforcement, Streaming Chunk Readers, and Semi-Structured Normalization

## Validation goals
- Enforce explicit data types (`dtypes`) and perform in-line date parsing (`parse_dates`) during CSV ingestion[cite: 11]
- Handle memory-constrained large files using iterative chunk processing (`chunksize`) and streaming calculations[cite: 11]
- Ingest and export multi-sheet Excel workbooks (`pd.ExcelWriter`, `sheet_name=None`)[cite: 11]
- Normalize nested, hierarchical JSON structures into clean 2D DataFrames using `pd.json_normalize()`[cite: 11]
- Evaluate memory footprints and operational drawbacks of default `object` type inference[cite: 11]

Data ingestion is the gateway to tabular data analysis. Pandas provides a suite of top-level `pd.read_*()` functions that parse unstructured or semi-structured files into in-memory DataFrames.

However, reading data naively can choke memory, slow down execution, or misinterpret data types. Understanding how to control chunk sizes, specify explicit schemas (`dtypes`), parse dates at load time, and handle multi-format I/O is critical when building efficient data pipelines.

### Core Concepts & Deep
 1. Dive1. Delimited Text Files: `pd.read_csv()`
    `CSV` (Comma-Separated Values) and `TSV` (Tab-Separated Values) are plain-text formats.
    - Schema Inference Overhead: By default, Pandas reads the first few rows to guess data types (`int64`, `float64`, `object`). For large files, passing `dtype={'column_name': 'category'}` skips type inference, saves memory, and speeds up loading.
    - Date Parsing: Converting date strings after loading using `pd.to_datetime()` requires extra memory allocations. Passing `parse_dates=['date_column']` inside `read_csv()` handles datetime parsing directly during string tokenization.
    - Chunking Large Datasets: When a dataset exceeds available RAM, set `chunksize=N`. This returns an TextFileReader iterator yielding DataFrames of $N$ rows at a time, allowing for streaming aggregations without loading the full file into memory.

 2. Spreadsheet Files: `pd.read_excel()`
   Excel files (`.xlsx`, .`xls`) are zip archives containing XML spreadsheets.
    - Engine Dependency: Pandas relies on open-source engines under the hood—openpyxl for .xlsx and xlrd for older .xls files.
    - Sheet Selection: Use sheet_name=None to load all sheets as a Python dictionary of DataFrames ({'Sheet1': df1, 'Sheet2': df2}), or pass a specific sheet name/index.    

  3. Semi-Structured Data: `pd.read_json()` & `json_normalize()`
   `JSON` (JavaScript Object Notation) is the standard format for web APIs and NoSQL databases.
      - Orientations: JSON files come in different structural layouts:
      records: Array of objects `[{"a": 1, "b": 2}, {"a": 3, "b": 4}]` (most common API output).
      - columns: Object of objects `{"a": {"0": 1, "1": 3}, "b": {"0": 2, "1": 4}}`.
      - Nested `JSON`: When `JSON` contains nested objects or lists (e.g., a user object containing an address dictionary), simple `pd.read_json()` flattens poorly. `pd.json_normalize()` recursively flattens nested `JSON` hierarchies into clean 2D tabular DataFrames.  

Here is a comprehensive hands-on script demonstrating file creation, optimized reading, chunk processing, sheet extraction, and nested JSON normalization.

In [2]:
import json
import numpy as np
import pandas as pd

# =====================================================================
# 1. Delimited Data Processing: Schema Definition & Streaming
# =====================================================================
np.random.seed(42)
n_rows = 10000

# Create dummy transactional dataset
mock_data = pd.DataFrame({
    "txn_id": [f"TXN_{i:06d}" for i in range(n_rows)],
    "timestamp": pd.date_range("2026-01-01", periods=n_rows, freq="min"),
    "store_code": np.random.choice(["STORE_A", "STORE_B", "STORE_C"], size=n_rows),
    "amount": np.round(np.random.uniform(10.0, 1000.0, size=n_rows), 2),
})
mock_data.to_csv("transactions.csv", index=False)

# OPTIMIZED READ: Explicit dtypes and direct date parsing
schema_definition = {
    "txn_id": "string",
    "store_code": "category",
    "amount": "float32"
}

df_csv = pd.read_csv(
    "transactions.csv",
    dtype=schema_definition,
    parse_dates=["timestamp"]
)

print("--- Optimized Ingestion Summary ---")
print(df_csv.info())
print("\nExact Memory Usage (Bytes):")
print(df_csv.memory_usage(deep=True))

# STREAMING CHUNK READ: Process file in batches of 2,500 rows
total_revenue = 0.0
for chunk in pd.read_csv("transactions.csv", chunksize=2500, usecols=["amount"]):
    total_revenue += chunk["amount"].sum()

print(f"\nStreaming Cumulative Revenue Calculation: ${total_revenue:,.2f}")


# =====================================================================
# 2. Excel Workbooks: Selective & Multi-Sheet Extraction
# =====================================================================
with pd.ExcelWriter("sales_report.xlsx", engine="openpyxl") as writer:
    df_csv[df_csv["store_code"] == "STORE_A"].to_excel(writer, sheet_name="Store_A", index=False)
    df_csv[df_csv["store_code"] == "STORE_B"].to_excel(writer, sheet_name="Store_B", index=False)

# Read all sheets into a dictionary of DataFrames
all_sheets = pd.read_excel("sales_report.xlsx", sheet_name=None)
print("\n--- Excel Workbook Sheets Ingested ---")
for name, sheet_df in all_sheets.items():
    print(f"Sheet Name: {name} | Dimensions: {sheet_df.shape}")


# =====================================================================
# 3. Semi-Structured JSON: Normalization & Structural Flattening
# =====================================================================
nested_api_payload = [
    {
        "order_id": "ORD_9001",
        "customer": {"id": "CUST_001", "tier": "Gold"},
        "line_items": [
            {"item_id": "SKU_A", "qty": 2, "unit_price": 150.0},
            {"item_id": "SKU_B", "qty": 1, "unit_price": 45.0}
        ]
    },
    {
        "order_id": "ORD_9002",
        "customer": {"id": "CUST_002", "tier": "Silver"},
        "line_items": [
            {"item_id": "SKU_C", "qty": 5, "unit_price": 12.5}
        ]
    }
]

# Unpack hierarchical list inside JSON payload using pd.json_normalize
df_flat = pd.json_normalize(
    nested_api_payload,
    record_path=["line_items"],
    meta=["order_id", ["customer", "id"], ["customer", "tier"]]
)

print("\n--- Flattened JSON API Data ---")
print(df_flat)

--- Optimized Ingestion Summary ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   txn_id      10000 non-null  string        
 1   timestamp   10000 non-null  datetime64[us]
 2   store_code  10000 non-null  category      
 3   amount      10000 non-null  float32       
dtypes: category(1), datetime64[us](1), float32(1), string(1)
memory usage: 205.3 KB
None

Exact Memory Usage (Bytes):
Index            132
txn_id        590000
timestamp      80000
store_code     10276
amount         40000
dtype: int64

Streaming Cumulative Revenue Calculation: $5,058,905.91

--- Excel Workbook Sheets Ingested ---
Sheet Name: Store_A | Dimensions: (3354, 4)
Sheet Name: Store_B | Dimensions: (3347, 4)

--- Flattened JSON API Data ---
  item_id  qty  unit_price  order_id customer.id customer.tier
0   SKU_A    2       150.0  ORD_9001    CUST_001          Gold

### Guided Question:

Suppose you are reading a 5GB CSV log containing millions of rows. One column represents status codes with three possible text values ("PENDING", "SUCCESS", "FAILED").

What dtype assignment strategy should you specify in pd.read_csv() to minimize memory usage?

What computational drawback does default object type inference introduce for this scenario?

1. Specify the category dtype
Since the status column contains only three distinct text values, use Pandas' category dtype when reading the CSV.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "logs.csv",
    dtype={"status": "category"}
)

Categorical data stores the distinct status strings once and represents individual rows using compact integer codes. This can significantly reduce memory usage when a column has millions of rows but very few unique values.

2. Drawback of default object inference
By default, Pandas may infer the status column as object, which typically stores references to Python string objects.
This introduces two main costs:
- Higher memory usage: Each row requires a reference to a string object, with additional overhead from the underlying Python strings.
- Slower processing: String-based operations can involve Python-level object handling and have more overhead than operations on compact categorical codes.

For a 5 GB `CSV` with millions of rows and only three possible statuses, specifying dtype=`{"status": "category"}` can substantially reduce the memory footprint of that column.
Production tip: If memory is still a concern, combine categorical types with `usecols` to load only required columns and chunksize to process the `CSV` in batches rather than loading the entire file into RAM.

### Guided Question:

When reading a dataset that exceeds available RAM using `pd.read_csv(file, chunksize=50000)`:

- What type of Python object does `pd.read_csv()` return when `chunksize` is specified?

- How would you write a loop using this iterator to calculate the global mean of a numerical column without storing all chunks in memory simultaneously?

1. When you specify chunksize=50000, pd.read_csv() returns a TextFileReader iterator, which yields a Pandas DataFrame containing up to 50,000 rows per chunk.

In [ ]:
reader = pd.read_csv("large_dataset.csv", chunksize=50000)

You can iterate over reader to process one chunk at a time instead of loading the entire dataset into RAM.

2. Use a running sum and running count. Do not average the individual chunk means directly, because chunks may contain different numbers of valid values.

In [ ]:
import pandas as pd

total_sum = 0.0
total_count = 0

for chunk in pd.read_csv("large_dataset.csv", chunksize=50000):
    values = chunk["numerical_column"]

    total_sum += values.sum()
    total_count += values.count()  # Excludes missing values (NaN)

global_mean = total_sum / total_count if total_count > 0 else float("nan")

print("Global mean:", global_mean)

Why this works
- `values.sum()` calculates the sum of the numerical values in the current chunk.
- `values.count()` counts only non-missing values.
- The running totals accumulate statistics across all chunks without retaining previously processed chunks.
- Dividing the total sum by the total count produces the global mean, excluding missing values.
Memory advantage: The calculation only needs the current chunk and a few scalar variables in memory, rather than the entire dataset.
For a very large CSV, you can also specify usecols=["numerical_column"] in `pd.read_csv()` to avoid loading unnecessary columns.

### Guided Question:

Given an API endpoint returning nested JSON objects where orders contain embedded customer metadata and a list of purchased items:
- Why does standard `pd.read_json()` fail to produce a clean 2D tabular representation for analysis?
- What parameters in `pd.json_normalize()` allow you to extract array elements as individual rows while duplicating parent metadata across those rows?

1. Standard `pd.read_json()` can read JSON data, but it does not automatically flatten arbitrarily nested objects and arrays into a clean, analysis-ready 2D table.
For example, an order might contain customer metadata and a list of purchased items. These nested structures can produce columns containing dictionaries or lists instead of individual scalar values.
For data analysis, you typically want one row per purchased item, with the relevant order and customer details repeated on each row.

2. Use `pd.json_normalize()` with record_path and meta
The two key parameters are:
- record_path: identifies the nested list whose elements should become individual rows.
- meta: specifies parent-level fields to extract and duplicate across those rows.

Use record_path to expand nested arrays into rows and meta to preserve parent metadata on every resulting row. Nested parent fields can be specified as lists of keys, such as `["customer", "name"]`

In [3]:
import pandas as pd

data = [
    {
        "order_id": 101,
        "customer": {
            "name": "Ahmad",
            "city": "Lagos"
        },
        "items": [
            {"product": "Laptop", "price": 500000},
            {"product": "Mouse", "price": 10000}
        ]
    }
]

df = pd.json_normalize(
    data,
    record_path="items",
    meta=[
        "order_id",
        ["customer", "name"],
        ["customer", "city"]
    ]
)

print(df)

  product   price order_id customer.name customer.city
0  Laptop  500000      101         Ahmad         Lagos
1   Mouse   10000      101         Ahmad         Lagos
